# Сдвинутые срезы как обучающие данные

In [ ]:
'''
Что уже известно:
- лучшая модель на train: 222 признака, LightGBM + CatBoost, 0.722 на кросс-валидации, 0.718 на Public;
- схема «история -> лаг 14 дней -> scoring_date -> target 3 месяца» одинакова для любой даты,
  поэтому target можно посчитать самим на более ранних датах по той же истории;
- раньше это направление было отвергнуто по нечестному сравнению, здесь оно проверяется как следует.

Что делаем здесь:
- для каждого клиента (train и test) строим до четырёх сдвинутых срезов: целевое окно целиком лежит
  в его известной истории; оставляем только строки без смещения отбора;
- чтобы признаки настоящих и сдвинутых срезов были определены одинаково, историю всех строк
  ограничиваем одной и той же глубиной HIST_DAYS;
- сравниваем на настоящих срезах train, по одним и тем же фолдам по клиентам:
    A — обучение только на train (как сейчас);
    B — обучение только на сдвинутых срезах, настоящие метки модель не видит вообще;
    C — train и сдвинутые срезы вместе, с признаком-флагом «сдвинутый срез»;
    D — train плюс прогноз модели B как дополнительный признак;
- проверяем, есть ли в сдвинутых срезах правило «последний заказ + средний интервал»;
- сохраняем отправки по вариантам; submission.csv — лучший по локальной проверке.
'''
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from IPython.display import display
from scipy.special import ndtr
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

KAGGLE_DIR = Path("/kaggle/input/competitions/lamoda-cu-repeat-purchase-2026")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else Path(os.environ.get("LAMODA_DATA_DIR", "data"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("LAMODA_OUT_DIR", "."))

SEED = 42
GAP_DAYS = 14
TARGET_MONTHS = 3
ACTIVE_DAYS = 365          # правило отбора: есть заказ за 365 дней до отсечения
SHIFTS = [0, 1, 2, 3]      # на сколько месяцев раньше отсечения заканчивается целевое окно сдвинутого среза
N_FOLDS = 5
WINDOWS = [7, 14, 30, 60, 90, 180, 365]


def average_precision_at_20_percent(customer_id, target, score):
    ranking = (
        pd.DataFrame({"customer_id": np.asarray(customer_id).astype(str), "target": np.asarray(target), "score": np.asarray(score)})
        .sort_values(["score", "customer_id"], ascending=[False, True], kind="mergesort")
        .reset_index(drop=True)
    )
    k = int(np.ceil(len(ranking) * 0.20))
    top_target = ranking["target"].iloc[:k].to_numpy()
    precision_at_rank = np.cumsum(top_target) / np.arange(1, k + 1)
    denominator = min(k, int(ranking["target"].sum()))
    return (precision_at_rank * top_target).sum() / max(denominator, 1)

## Данные и границы сдвинутых срезов

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["scoring_date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["scoring_date"])
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
orders = pd.read_parquet(DATA_DIR / "orders.parquet")
products = pd.read_parquet(DATA_DIR / "products.parquet")
orders["order_date"] = pd.to_datetime(orders["order_date"])

cust = pd.concat([train.assign(part="train"), test.assign(part="test", target=np.nan)], ignore_index=True)
cust["slice"] = cust["scoring_date"].dt.strftime("%Y-%m-%d")
cust["cutoff"] = cust["scoring_date"] - pd.Timedelta(days=GAP_DAYS)
cust["win_len"] = cust["scoring_date"].map(lambda d: (d + pd.DateOffset(months=TARGET_MONTHS) - d).days)
cust = cust.set_index("customer_id")

lines = orders.merge(products, on="product_id", how="left", validate="m:1")
lines["cutoff"] = lines["customer_id"].map(cust["cutoff"])
lines["d"] = (lines["cutoff"] - lines["order_date"]).dt.days  # дней от заказа до настоящего отсечения клиента
assert lines["d"].between(0, 732).all()


def shift_bounds(k):
    # Сдвинутый срез: целевое окно (p_date, p_end], p_end = отсечение − k месяцев; история — до p_cut включительно.
    t = pd.DataFrame({"cutoff": sorted(cust["cutoff"].unique())})
    t["p_end"] = t["cutoff"] - pd.DateOffset(months=k)
    t["p_date"] = t["p_end"] - pd.DateOffset(months=TARGET_MONTHS)
    t["p_cut"] = t["p_date"] - pd.Timedelta(days=GAP_DAYS)
    t["off"] = (t["cutoff"] - t["p_cut"]).dt.days        # на сколько дней сдвинуто отсечение
    t["lab_lo"] = (t["cutoff"] - t["p_end"]).dt.days     # целевое окно в днях до настоящего отсечения: [lab_lo, lab_hi)
    t["lab_hi"] = (t["cutoff"] - t["p_date"]).dt.days
    t["win"] = (t["p_end"] - t["p_date"]).dt.days
    return t.set_index("cutoff")


BOUNDS = {k: shift_bounds(k) for k in SHIFTS}
HIST_DAYS = 730 - max(int(b["off"].max()) for b in BOUNDS.values())  # общая глубина истории для всех строк
N_MB = HIST_DAYS // 30 + 1
print("глубина истории для всех строк, дней:", HIST_DAYS)
display(pd.concat({k: b[["p_cut", "p_date", "p_end", "off", "win"]] for k, b in BOUNDS.items()}))

## Признаки

In [ ]:
def clean(name):
    return re.sub(r"\W+", "_", str(name)).strip("_").lower()


def build_features(lines, win_len):
    lines = lines.copy()
    lines["has_disc"] = (lines["discount_percent"] > 0).astype("float32")
    lines["has_loy"] = (lines["loyalty_percent"] > 0).astype("float32")
    lines["weekend"] = (lines["order_date"].dt.dayofweek >= 5).astype("float32")

    # --- уровень заказа, самые свежие заказы идут первыми
    ot = (
        lines.groupby(["customer_id", "order_id"], sort=False)
        .agg(
            d=("d", "first"),
            order_sum=("paid_price", "sum"),
            n_items=("paid_price", "size"),
            disc=("discount_percent", "mean"),
            loy=("loyalty_percent", "max"),
            weekend=("weekend", "first"),
        )
        .reset_index()
        .sort_values(["customer_id", "d", "order_id"])
        .reset_index(drop=True)
    )
    cid = ot["customer_id"]
    f = ot.groupby("customer_id").agg(
        recency=("d", "min"),
        tenure=("d", "max"),
        d_mean=("d", "mean"),
        d_std=("d", "std"),
        n_orders=("order_id", "size"),
        n_items=("n_items", "sum"),
        sum_paid=("order_sum", "sum"),
        order_sum_mean=("order_sum", "mean"),
        order_sum_median=("order_sum", "median"),
        order_sum_max=("order_sum", "max"),
        order_sum_std=("order_sum", "std"),
        items_per_order=("n_items", "mean"),
        items_max=("n_items", "max"),
        order_disc_mean=("disc", "mean"),
        order_loy_mean=("loy", "mean"),
        order_loy_max=("loy", "max"),
        weekend_share=("weekend", "mean"),
    )

    last = ot.drop_duplicates("customer_id").set_index("customer_id")
    first = ot.drop_duplicates("customer_id", keep="last").set_index("customer_id")
    f["last_order_sum"] = last["order_sum"]
    f["last_n_items"] = last["n_items"]
    f["last_disc"] = last["disc"]
    f["last_loy"] = last["loy"]
    f["first_loy"] = first["loy"]
    f["first_order_sum"] = first["order_sum"]
    f["loy_change"] = f["last_loy"] - f["first_loy"]
    f["last_sum_to_mean"] = f["last_order_sum"] / f["order_sum_mean"]

    # --- окна по времени
    for w in WINDOWS:
        m = ot["d"] <= w
        f[f"n_orders_{w}"] = m.groupby(cid).sum()
        f[f"sum_{w}"] = (ot["order_sum"] * m).groupby(cid).sum()
        f[f"n_items_{w}"] = (ot["n_items"] * m).groupby(cid).sum()
    for w in [90, 365]:
        m = ot["d"] <= w
        f[f"loy_max_{w}"] = ot["loy"].where(m).groupby(cid).max()
        f[f"disc_mean_{w}"] = ot["disc"].where(m).groupby(cid).mean()
        f[f"aov_{w}"] = f[f"sum_{w}"] / f[f"n_orders_{w}"].replace(0, np.nan)
    f["n_orders_older_365"] = f["n_orders"] - f["n_orders_365"]

    # --- уровень дня: ритм заказов
    days = ot.groupby(["customer_id", "d"], as_index=False).agg(day_orders=("order_id", "size"), day_sum=("order_sum", "sum"))
    days["gap"] = days.groupby("customer_id")["d"].diff()
    days["rk"] = days.groupby("customer_id").cumcount()
    gd = days.groupby("customer_id")
    f["n_days"] = gd.size()
    f["max_day_orders"] = gd["day_orders"].max()
    f["multi_order_days"] = (days["day_orders"] > 1).groupby(days["customer_id"]).sum()
    f = f.join(gd["gap"].agg(gap_mean="mean", gap_median="median", gap_std="std", gap_min="min", gap_max="max"))
    for k in [1, 2, 4, 9]:
        f[f"d_day_{k + 1}"] = days.loc[days["rk"] == k].set_index("customer_id")["d"]  # давность (k+1)-го с конца дня с заказом
    f["gap_prev"] = days.loc[days["rk"] == 2].set_index("customer_id")["gap"]
    for name, period, limit in [("months_12", 30, 12), ("months_24", 30, 25), ("weeks_13", 7, 13), ("weeks_26", 7, 26)]:
        b = days.assign(b=days["d"] // period)
        f[f"active_{name}"] = b.loc[b["b"] < limit].drop_duplicates(["customer_id", "b"]).groupby("customer_id").size()
    f[[c for c in f.columns if c.startswith("active_")]] = f[[c for c in f.columns if c.startswith("active_")]].fillna(0)

    # --- попадание следующего заказа в целевое окно
    # Время считаем в днях после отсечения: окно — это (GAP_DAYS, GAP_DAYS + win_len].
    r = f["recency"]
    w0, w1 = GAP_DAYS, GAP_DAYS + win_len.reindex(f.index)
    dc = days["customer_id"]
    g, dr = days["gap"], dc.map(r)
    lo, hi = dr + GAP_DAYS, dr + GAP_DAYS + dc.map(win_len)
    n_gaps = g.notna().groupby(dc).sum()
    alive = (g > dr).groupby(dc).sum()  # сколько собственных интервалов клиента длиннее текущей паузы
    f["own_gaps_gt_recency"] = alive
    f["own_share_gaps_gt_recency"] = alive / n_gaps.replace(0, np.nan)
    f["own_p_hit"] = ((g > lo) & (g <= hi)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_hidden"] = ((g > dr) & (g <= lo)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_late"] = (g > hi).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_share_gaps_le_window"] = (g <= dc.map(win_len)).groupby(dc).sum() / n_gaps.replace(0, np.nan)

    wgt = 0.7 ** (days["rk"] - 1).clip(lower=0)
    f["gap_ewm"] = (g * wgt).groupby(dc).sum() / wgt.where(g.notna()).groupby(dc).sum()
    f["gap_last3"] = g.where(days["rk"].between(1, 3)).groupby(dc).mean()
    f["gap_last"] = days.loc[days["rk"] == 1].set_index("customer_id")["gap"]
    f["gap_last_to_median"] = f["gap_last"] / f["gap_median"]
    for name in ["gap_last", "gap_last3", "gap_ewm", "gap_median", "gap_mean", "gap_min", "gap_max"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r  # ожидаемый день следующего заказа; отрицательный — клиент «опаздывает»
        f[f"tnext_{short}"] = t
        f[f"tnext_{short}_dist"] = (w0 - t).clip(lower=0) + (t - w1).clip(lower=0)  # расстояние до окна, 0 — внутри
        if name in ["gap_last", "gap_ewm", "gap_median", "gap_mean"]:
            f[f"cycles_{short}"] = np.floor((r + w1) / gg) - np.floor((r + w0) / gg)  # сколько раз период укладывается в окно
            f[f"tmod_{short}"] = gg - (r % gg)  # до ближайшего кратного периоду дня, если клиент пропускал циклы

    # --- окно в единицах собственного периода и сглаженное попадание
    for name in ["gap_mean", "gap_median", "gap_last", "gap_last3"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r
        f[f"ulo_{short}"] = (r + w0) / gg  # следующий заказ попадает в окно, если ulo < 1 <= uhi
        f[f"uhi_{short}"] = (r + w1) / gg
        for sd in (10, 20):  # день следующего заказа известен с точностью ± sd дней
            f[f"phit{sd}_{short}"] = ndtr((w1 - t) / sd) - ndtr((w0 - t) / sd)
        f[f"phitrel_{short}"] = ndtr((w1 - t) / (0.25 * gg)) - ndtr((w0 - t) / (0.25 * gg))

    # --- дни с заказами по 30-дневным интервалам и в том же окне год и два назад
    mb = pd.crosstab(dc, (days["d"] // 30).clip(upper=N_MB)).reindex(columns=range(N_MB), fill_value=0)
    mb.columns = [f"mb_{i:02d}" for i in mb.columns]
    f = f.join(mb)
    dl = dc.map(win_len)
    for years, tag in [(1, "ya")]:
        start, end = 365 * years - GAP_DAYS - dl, 365 * years - GAP_DAYS - 1
        f[f"{tag}_days"] = ((days["d"] >= start) & (days["d"] <= end)).groupby(dc).sum()
        f[f"{tag}_days_wide"] = ((days["d"] >= start - 30) & (days["d"] <= end + 30)).groupby(dc).sum()

    # --- производные
    f["formula"] = f["n_orders"] / (f["recency"] + 30)
    f["days_formula"] = f["n_days"] / (f["recency"] + 30)
    f["orders_per_30d"] = f["n_orders"] / (f["tenure"] + 30) * 30
    f["recency_to_gap_mean"] = f["recency"] / f["gap_mean"]
    f["recency_to_gap_median"] = f["recency"] / f["gap_median"]
    f["recency_minus_gap_max"] = f["recency"] - f["gap_max"]
    f["gap_cv"] = f["gap_std"] / f["gap_mean"]
    f["recency_to_tenure"] = f["recency"] / (f["tenure"] + 1)
    f["share_orders_30"] = f["n_orders_30"] / f["n_orders"]
    f["share_orders_90"] = f["n_orders_90"] / f["n_orders"]
    f["share_orders_365"] = f["n_orders_365"] / f["n_orders"]
    f["trend_90_vs_365"] = f["n_orders_90"] / (f["n_orders_365"] - f["n_orders_90"] + 1)
    f["trend_30_vs_90"] = f["n_orders_30"] / (f["n_orders_90"] - f["n_orders_30"] + 1)
    f["share_sum_90"] = f["sum_90"] / f["sum_paid"].replace(0, np.nan)

    # --- уровень позиции: цены, скидки, лояльность, состав покупок
    gl = lines.groupby("customer_id")
    f = f.join(
        gl.agg(
            price_mean=("paid_price", "mean"),
            price_median=("paid_price", "median"),
            price_max=("paid_price", "max"),
            price_min=("paid_price", "min"),
            disc_mean=("discount_percent", "mean"),
            disc_max=("discount_percent", "max"),
            disc_share=("has_disc", "mean"),
            loy_mean=("loyalty_percent", "mean"),
            loy_share=("has_loy", "mean"),
            n_loy_values=("loyalty_percent", "nunique"),
            n_products=("product_id", "nunique"),
            n_brands=("brand", "nunique"),
            n_cat=("category", "nunique"),
            n_cat2=("category2", "nunique"),
            n_gender=("gender_product", "nunique"),
        )
    )
    f["repeat_product_share"] = 1 - f["n_products"] / f["n_items"]
    f["brands_per_item"] = f["n_brands"] / f["n_items"]
    f["cat2_per_item"] = f["n_cat2"] / f["n_items"]

    for col, prefix in [("category", "cat"), ("gender_product", "gender")]:
        share = pd.crosstab(lines["customer_id"], lines[col], normalize="index")
        share.columns = [f"{prefix}_share_{clean(c)}" for c in share.columns]
        f = f.join(share)
        recent = lines.loc[lines["d"] <= 90]
        share = pd.crosstab(recent["customer_id"], recent[col], normalize="index")
        share.columns = [f"{prefix}_share90_{clean(c)}" for c in share.columns]
        f = f.join(share)

    for col in ["category2", "brand"]:
        top = (
            lines.groupby(["customer_id", col], observed=True).size().rename("n").reset_index()
            .sort_values(["customer_id", "n", col], ascending=[True, False, True])
            .drop_duplicates("customer_id").set_index("customer_id")
        )
        f[f"top_{col}_share"] = top["n"] / f["n_items"]
        if col == "category2":
            f["top_category2"] = top[col].astype(str)

    # --- ритм отдельно по категориям, подкатегориям и брендам
    for col in ["category", "category2", "brand"]:
        st = (
            lines[["customer_id", col, "d"]].drop_duplicates()
            .groupby(["customer_id", col], observed=True)["d"].agg(n="size", last="min", first="max")
        )
        st = st[st["n"] >= 2].reset_index()
        st["gap"] = (st["first"] - st["last"]) / (st["n"] - 1)
        st["t"] = st["gap"] - st["last"]
        hi = GAP_DAYS + st["customer_id"].map(win_len)
        st["dist"] = (GAP_DAYS - st["t"]).clip(lower=0) + (st["t"] - hi).clip(lower=0)
        st["hit"] = (st["dist"] == 0).astype(float)
        gs = st.groupby("customer_id")
        f[f"ser_{col}_n"] = gs.size()
        f[f"ser_{col}_hit"] = gs["hit"].sum()
        f[[f"ser_{col}_n", f"ser_{col}_hit"]] = f[[f"ser_{col}_n", f"ser_{col}_hit"]].fillna(0)
        f[f"ser_{col}_hit_share"] = gs["hit"].mean()
        f[f"ser_{col}_min_dist"] = gs["dist"].min()
        best = st.sort_values(["customer_id", "dist", "last"]).drop_duplicates("customer_id").set_index("customer_id")
        f[f"ser_{col}_best_t"] = best["t"]
        f[f"ser_{col}_best_gap"] = best["gap"]

    return f.replace([np.inf, -np.inf], np.nan)


def make_rows(k):
    # k = None — настоящий срез; иначе сдвинутый срез с номером k
    if k is None:
        hist = lines[lines["d"] <= HIST_DAYS]
        f = build_features(hist, cust["win_len"])
        f["y"] = cust["target"].reindex(f.index)
        f["shift"], f["is_shifted"] = -1, 0
        return f
    b = BOUNDS[k]
    off = lines["cutoff"].map(b["off"])
    in_target = (lines["d"] >= lines["cutoff"].map(b["lab_lo"])) & (lines["d"] < lines["cutoff"].map(b["lab_hi"]))
    buyers = set(lines.loc[in_target, "customer_id"])
    hist = lines.assign(d=lines["d"] - off)
    hist = hist[hist["d"].between(0, HIST_DAYS)]
    f = build_features(hist, cust["cutoff"].map(b["win"]))
    f["y"] = f.index.isin(buyers).astype(float)
    f["shift"], f["is_shifted"] = k, 1
    # без смещения отбора: последний заказ на сдвинутую дату уже лежит в окне 365 дней до настоящего отсечения
    keep = f["recency"] + cust["cutoff"].map(b["off"]).reindex(f.index) <= ACTIVE_DAYS
    return f[keep]


D = pd.concat([make_rows(None)] + [make_rows(k) for k in SHIFTS]).reset_index()
D["part"] = D["customer_id"].map(cust["part"])
D["slice"] = D["customer_id"].map(cust["slice"])
META = ["customer_id", "part", "slice", "shift", "is_shifted", "y"]
FEATURES = [c for c in D.columns if c not in META]
CAT_COLS = ["top_category2"]
D[CAT_COLS] = D[CAT_COLS].astype("category")

real_tr = D[(D["shift"] == -1) & D["part"].eq("train")].reset_index(drop=True)
real_te = D[(D["shift"] == -1) & D["part"].eq("test")].set_index("customer_id").loc[sample["customer_id"]].reset_index()
shifted = D[D["is_shifted"] == 1].reset_index(drop=True)
assert len(real_tr) == len(train) and len(real_te) == len(test)
print("признаков:", len(FEATURES))
display(
    D.assign(срез=np.where(D["is_shifted"] == 1, "сдвиг " + D["shift"].astype(str), "настоящий"))
    .groupby(["срез", "part"])["y"].agg(строк="size", доля_y="mean")
)

## Есть ли правило в сдвинутых срезах

In [ ]:
N_EDGES = [0, 1, 2, 3, 5, 9, 20, 10_000]
N_LABELS = ["1", "2", "3", "4-5", "6-9", "10-20", "21+"]
t_fine = list(range(-20, 131, 10))
cmp_rule, cmp_zone = {}, {}
for name, df in [("настоящий train", real_tr), ("сдвинутые срезы", shifted)]:
    two = df[df["n_days"].eq(2)]
    cmp_rule[name] = two.groupby(pd.cut(two["tnext_mean"], t_fine), observed=False)["y"].mean()
    cmp_rule[name + ", строк"] = two.groupby(pd.cut(two["tnext_mean"], t_fine), observed=False)["y"].size()
    multi = df[df["n_days"].ge(2)].assign(n_bin=lambda t: pd.cut(t["n_days"], N_EDGES, labels=N_LABELS))
    inside = multi["tnext_mean"].gt(25) & multi["tnext_mean"].le(92)
    cmp_zone[name + ": в зоне"] = multi[inside].groupby("n_bin", observed=True)["y"].mean()
    cmp_zone[name + ": вне зоны"] = multi[~inside].groupby("n_bin", observed=True)["y"].mean()
print("Два дня с заказами: доля y = 1 по ожидаемому дню следующего заказа")
display(pd.DataFrame(cmp_rule).T)
print("Доля y = 1 в зоне 25..92 и вне её, по числу дней с заказами:")
display(pd.DataFrame(cmp_zone).T)

REC_EDGES = [-1, 7, 14, 30, 60, 90, 180, 365]
print("Доля y = 1: давность x число дней с заказами, настоящий train:")
display(real_tr.assign(r=pd.cut(real_tr["recency"], REC_EDGES), n=pd.cut(real_tr["n_days"], N_EDGES, labels=N_LABELS)).pivot_table(index="r", columns="n", values="y", observed=True))
print("То же, сдвинутые срезы:")
display(shifted.assign(r=pd.cut(shifted["recency"], REC_EDGES), n=pd.cut(shifted["n_days"], N_EDGES, labels=N_LABELS)).pivot_table(index="r", columns="n", values="y", observed=True))
print("То же, сдвинутые срезы отдельно по сдвигам (все группы вместе, только давность):")
display(shifted.assign(r=pd.cut(shifted["recency"], REC_EDGES)).pivot_table(index="r", columns="shift", values="y", observed=True))

## Сравнение вариантов обучения

In [ ]:
LGB_PARAMS = dict(
    objective="binary", learning_rate=0.05, num_leaves=31, min_child_samples=100,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=5.0, cat_smooth=50,
    n_estimators=3000, random_state=SEED, n_jobs=-1, verbose=-1,
)
EARLY_STOP = 150


def fit_predict(X_tr, y_tr, X_va, y_va, X_out):
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="binary_logloss", callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    return [model.predict_proba(X)[:, 1] for X in X_out], model.best_iteration_, pd.Series(model.booster_.feature_importance("gain"), index=X_tr.columns)


y_real = real_tr["y"].astype(int)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_of = pd.Series(-1, index=cust.index)   # клиенты теста остаются с -1 и всегда идут в обучение
for fold, (_, i_va) in enumerate(skf.split(real_tr, real_tr["slice"] + "_" + y_real.astype(str))):
    fold_of.loc[real_tr["customer_id"].iloc[i_va]] = fold
real_fold = real_tr["customer_id"].map(fold_of).to_numpy()
shift_fold = shifted["customer_id"].map(fold_of).to_numpy()

VARIANTS = ["A: только train", "B: только сдвинутые", "C: train + сдвинутые", "D: train + прогноз B"]
oof = {v: np.zeros(len(real_tr)) for v in VARIANTS}
pred_te = {v: np.zeros(len(real_te)) for v in VARIANTS}
iters = {v: [] for v in VARIANTS}
imp_c = []
F_C = FEATURES + ["is_shifted"]
F_D = FEATURES + ["p_shifted"]

for fold in range(N_FOLDS):
    va, tr_ = real_fold == fold, real_fold != fold
    s_tr, s_va = shift_fold != fold, shift_fold == fold

    # A: только настоящие метки
    (p_va, p_te), it, _ = fit_predict(real_tr.loc[tr_, FEATURES], y_real[tr_], real_tr.loc[va, FEATURES], y_real[va], [real_tr.loc[va, FEATURES], real_te[FEATURES]])
    oof[VARIANTS[0]][va], iters[VARIANTS[0]] = p_va, iters[VARIANTS[0]] + [it]
    pred_te[VARIANTS[0]] += p_te / N_FOLDS

    # B: только сдвинутые срезы; ранняя остановка по сдвинутым срезам отложенных клиентов, настоящие метки не используются
    (p_va, p_trn, p_te), it, _ = fit_predict(shifted.loc[s_tr, FEATURES], shifted.loc[s_tr, "y"], shifted.loc[s_va, FEATURES], shifted.loc[s_va, "y"],
                                             [real_tr.loc[va, FEATURES], real_tr.loc[tr_, FEATURES], real_te[FEATURES]])
    oof[VARIANTS[1]][va], iters[VARIANTS[1]] = p_va, iters[VARIANTS[1]] + [it]
    pred_te[VARIANTS[1]] += p_te / N_FOLDS

    # C: всё вместе, модель знает, какая строка сдвинутая
    X_c = pd.concat([real_tr.loc[tr_, F_C], shifted.loc[s_tr, F_C]], ignore_index=True)
    y_c = pd.concat([y_real[tr_], shifted.loc[s_tr, "y"]], ignore_index=True)
    (p_va, p_te), it, imp = fit_predict(X_c, y_c, real_tr.loc[va, F_C], y_real[va], [real_tr.loc[va, F_C], real_te[F_C]])
    oof[VARIANTS[2]][va], iters[VARIANTS[2]] = p_va, iters[VARIANTS[2]] + [it]
    pred_te[VARIANTS[2]] += p_te / N_FOLDS
    imp_c.append(imp / imp.sum())

    # D: настоящие метки плюс прогноз модели B как признак (модель B настоящих меток не видела, утечки нет)
    X_tr_d = real_tr.loc[tr_, FEATURES].assign(p_shifted=p_trn)
    X_va_d = real_tr.loc[va, FEATURES].assign(p_shifted=oof[VARIANTS[1]][va])
    X_te_d = real_te[FEATURES].assign(p_shifted=p_te)
    (p_va, p_te_d), it, _ = fit_predict(X_tr_d[F_D], y_real[tr_], X_va_d[F_D], y_real[va], [X_va_d[F_D], X_te_d[F_D]])
    oof[VARIANTS[3]][va], iters[VARIANTS[3]] = p_va, iters[VARIANTS[3]] + [it]
    pred_te[VARIANTS[3]] += p_te_d / N_FOLDS
    print(f"fold {fold} готов")


def rank_blend(preds):
    return np.mean([pd.Series(p).rank(pct=True).to_numpy() for p in preds], axis=0)


oof["смесь A + C"] = rank_blend([oof[VARIANTS[0]], oof[VARIANTS[2]]])
pred_te["смесь A + C"] = rank_blend([pred_te[VARIANTS[0]], pred_te[VARIANTS[2]]])
oof["смесь A + B"] = rank_blend([oof[VARIANTS[0]], oof[VARIANTS[1]]])
pred_te["смесь A + B"] = rank_blend([pred_te[VARIANTS[0]], pred_te[VARIANTS[1]]])

slices = sorted(real_tr["slice"].unique())
score_rows, bucket_rows = {}, {}
for name, p in oof.items():
    per = {s: average_precision_at_20_percent(real_tr.loc[real_tr["slice"].eq(s), "customer_id"], y_real[real_tr["slice"].eq(s)], p[real_tr["slice"].eq(s).to_numpy()]) for s in slices}
    per_fold = [np.mean([average_precision_at_20_percent(real_tr.loc[m, "customer_id"], y_real[m], p[m]) for m in [(real_fold == f) & real_tr["slice"].eq(s).to_numpy() for s in slices]]) for f in range(N_FOLDS)]
    score_rows[name] = {**per, "среднее": np.mean(list(per.values())), "разброс по фолдам": np.std(per_fold)}
    pct = pd.Series(p).groupby(real_tr["slice"]).rank(pct=True, ascending=False)
    bucket_rows[name] = y_real.groupby(pd.cut(pct, [0, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]), observed=True).mean()
scores = pd.DataFrame(score_rows).T
print("AP@20% на настоящих срезах train (прогнозы вне фолда):")
display(scores)
base = VARIANTS[0]
fold_ap = lambda p: np.array([np.mean([average_precision_at_20_percent(real_tr.loc[m, "customer_id"], y_real[m], p[m]) for m in [(real_fold == f) & real_tr["slice"].eq(s).to_numpy() for s in slices]]) for f in range(N_FOLDS)])
base_folds = fold_ap(oof[base])
print("Разница с вариантом A по фолдам (среднее ± разброс, в скольких фолдах плюс):")
for name, p in oof.items():
    if name != base:
        d = fold_ap(p) - base_folds
        print(f"  {name}: {d.mean():+.4f} ± {d.std():.4f}, плюс в {int((d > 0).sum())} из {N_FOLDS}")
print("Доля target = 1 по положению в рейтинге:")
display(pd.DataFrame(bucket_rows).T)
print("Итераций до остановки:", {k: [int(i) for i in v] for k, v in iters.items()})
print("Корреляция прогнозов A и B на train (Спирмен):", round(float(pd.Series(oof[VARIANTS[0]]).corr(pd.Series(oof[VARIANTS[1]]), method="spearman")), 4))
imp = pd.concat(imp_c, axis=1).mean(axis=1).sort_values(ascending=False)
print("Важность в варианте C: признак-флаг на месте", int(list(imp.index).index("is_shifted")) + 1, "из", len(imp), "| доля", round(float(imp["is_shifted"]), 4))

## Отправки

In [ ]:
best = scores["среднее"].idxmax()
print("Лучший вариант по локальной проверке:", best)
files = {"A: только train": "submission_A_train.csv", "B: только сдвинутые": "submission_B_shifted.csv", "C: train + сдвинутые": "submission_C_both.csv",
         "D: train + прогноз B": "submission_D_stack.csv", "смесь A + C": "submission_blend_AC.csv", "смесь A + B": "submission_blend_AB.csv"}
for name, fname in files.items():
    sub = pd.DataFrame({"customer_id": real_te["customer_id"].to_numpy(), "score": pd.Series(pred_te[name]).rank(pct=True).to_numpy()})
    assert sub["customer_id"].tolist() == sample["customer_id"].tolist() and sub["score"].between(0, 1).all()
    sub.to_csv(OUT_DIR / fname, index=False)
    if name == best:
        sub.to_csv(OUT_DIR / "submission.csv", index=False)
print("Корреляция прогнозов на тесте с вариантом A (Спирмен):")
print({name: round(float(pd.Series(p).corr(pd.Series(pred_te[base]), method="spearman")), 4) for name, p in pred_te.items() if name != base})